This notebook looks through the FTW benchmark catalog to find the 2024 collections that have HCAT crop labels, checks that their files contain the data we need, and saves that list of collections (and other settings).

In [ ]:
import json
import fsspec
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import urlopen
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.max_rows',10)

In [ ]:
CATALOGURL  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
INTERIMDIR  = Path('../data/interim')
REMOTE      = fsspec.filesystem('https')
COLLECTION  = 'ee/ee_2024'
YEAR        = 2024
MINFIELDS   = 100
MINCROPS    = 25
NWORKERS    = 16
FILECOLUMNS = {'fields':['id','hcat:code','hcat:name','geometry'],
               'field_embeddings':['id','pixels','pixel_rule','aef_mean','aef_std','olmoearth_mean','olmoearth_std'],
               'chips':['id','split','geometry']}

In [ ]:
def list_child_urls(catalogurl):
    with urlopen(catalogurl,timeout=60) as response:
        links = json.load(response)['links']
    return [urljoin(catalogurl,link['href']) for link in links if link['rel']=='child']

def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def get_parquet_metadata(fileurl):
    with REMOTE.open(fileurl,block_size=2**20) as file:
        parquet = pq.ParquetFile(file)
        return parquet.schema_arrow.names,parquet.metadata.num_rows

def load_columns(fileurl,cols):
    with REMOTE.open(fileurl) as file:
        return pq.read_table(file,columns=cols).to_pandas()

def summarize_collection_labels(collection):
    fieldsurl  = build_file_url(collection,'fields')
    cols,nrows = get_parquet_metadata(fieldsurl)
    if 'hcat:code' not in cols:
        return dict(collection=collection,nfields=nrows,haslabels=False)
    counts = load_columns(fieldsurl,['hcat:code'])['hcat:code'].value_counts()
    with urlopen(urljoin(CATALOGURL,f'{collection}/collection.json'),timeout=60) as response:
        name = json.load(response)['title'].split(' — ')[0]
    return dict(collection=collection,
                name=name,
                nfields=nrows,
                haslabels=True,
                pctlabelled=100*counts.sum()/nrows,
                nclasses=len(counts),
                nkept=(counts>=MINFIELDS).sum(),
                embedmb=REMOTE.size(build_file_url(collection,'field_embeddings'))/1e6)

In [ ]:
sourceurls = list_child_urls(CATALOGURL)

with ThreadPoolExecutor(NWORKERS) as executor:
    collectionurls = [url for urls in executor.map(list_child_urls,sourceurls) for url in urls]
allcollections  = ['/'.join(url.split('/')[-3:-1]) for url in collectionurls]
yearcollections = [collection for collection in allcollections if collection.endswith(str(YEAR))]

print('The FTW benchmark has...')
print(f'- {len(sourceurls)} sources (a dataset of fields from one region, like `ee`)')
print(f'- {len(allcollections)} collections (one year of one source, like `ee_2024`)')
print(f'- {len(yearcollections)} sources have {YEAR} data ({len(yearcollections)} collections)')

In [ ]:
with ThreadPoolExecutor(NWORKERS) as executor:
    summaries = pd.DataFrame(list(executor.map(summarize_collection_labels,yearcollections))).sort_values('nfields').reset_index(drop=True)
labelled    = summaries[summaries.haslabels].drop(columns='haslabels').astype({'nclasses':int,'nkept':int}).reset_index(drop=True)
unlabelled  = summaries.loc[~summaries.haslabels,'collection']
coarse      = labelled.loc[labelled.nkept<MINCROPS,'collection']
selected    = labelled[labelled.nkept>=MINCROPS].reset_index(drop=True)
COLLECTIONS = selected.collection.tolist()
headers     = {'collection':'Collection','nfields':'Fields','pctlabelled':'Labelled Fields (%)','nclasses':'Crop Classes',
               'nkept':f'Crop Classes with ≥ {MINFIELDS} Fields','embedmb':'Embeddings File Size (MB)'}

print(f'Of the {len(summaries)} collections from {YEAR}...')
print(f'- {len(unlabelled)} have no HCAT crop labels')
print(f'- {len(labelled)} have HCAT crop labels')
print(f"- {len(coarse)} of those have fewer than {MINCROPS} crops with ≥ {MINFIELDS} fields, so their labels are too coarse to use: {', '.join(coarse)}")
print(f'- {len(selected)} are kept, totaling {selected.nfields.sum():,} fields and {selected.embedmb.sum()/1e3:,.1f} GB of embeddings')

In [ ]:
display(labelled.rename(columns=headers).style.format(
    {'Fields':'{:,}','Labelled Fields (%)':'{:.1f}','Embeddings File Size (MB)':'{:,.0f}'}).hide(axis='index'))

In [ ]:
fields  = load_columns(build_file_url(COLLECTION,'fields'),['hcat:code','hcat:name'])
crops   = fields.value_counts(['hcat:code','hcat:name']).rename('nfields').reset_index()
crops['pct'] = 100*crops.nfields/len(fields)
kept    = crops[crops.nfields>=MINFIELDS]
headers = {'hcat:code':'HCAT Code','hcat:name':'HCAT Class','nfields':'Fields','pct':'Fields (%)'}

display(kept.rename(columns=headers).style.format({'Fields':'{:,}','Fields (%)':'{:.2f}'}).hide(axis='index'))

In [ ]:
pairs = [(collection,filetype) for collection in COLLECTIONS for filetype in FILECOLUMNS]

with ThreadPoolExecutor(NWORKERS) as executor:
    columns = list(executor.map(lambda pair:get_parquet_metadata(build_file_url(*pair))[0],pairs))
missing = {pair:sorted(set(FILECOLUMNS[pair[1]])-set(cols)) for pair,cols in zip(pairs,columns) if set(FILECOLUMNS[pair[1]])-set(cols)}

print(f'{len(pairs)-len(missing)} of {len(pairs)} files ({len(COLLECTIONS)} collections × {len(FILECOLUMNS)} files each) have all columns needed!')

In [ ]:
selected.to_csv(INTERIMDIR/'collections.csv',index=False)
(INTERIMDIR/'settings.json').write_text(json.dumps({'year':YEAR,'minfields':MINFIELDS,'mincrops':MINCROPS,'filecolumns':FILECOLUMNS},indent=2))

print(f"Saved {INTERIMDIR/'collections.csv'}: which collections to use (the {len(selected)} from {YEAR} with detailed HCAT crop labels)")
print(f"Saved {INTERIMDIR/'settings.json'}: the settings shared throughout the analysis (YEAR, MINFIELDS, MINCROPS, and the columns to read from each file)")